In [31]:
from collections import OrderedDict
from csv import DictWriter
from datetime import timedelta, timezone, datetime
from typing import Iterable

import bs4
import requests

In [32]:
# In the Environment Canada API days end when they end in PST
pst = timezone(timedelta(hours=-8))  # UTC -8:00
one_day = timedelta(days=1)
# The APIs do not present data for today until the day is complete.
# As a result we look for stations with data yesterday
today = datetime.now(tz=pst)
yesterday = today - one_day
y_year = yesterday.year
y_month = yesterday.month
y_day = yesterday.day

host = 'https://climate.weather.gc.ca'

In [33]:
def generate_stations(api_host: str = host, year: int = y_year, month: int = y_month, day: int = y_day) \
        -> Iterable[int]:
    """Returns every active station ID one at time"""

    start = 0
    interval = 100
    while True:
        station_list_url = f'{api_host}/historical_data/search_historic_data_stations_e.html?' \
                           f'searchType=stnProv&' \
                           f'timeframe=1' \
                           f'&lstProvince=' \
                           f'&StartYear=1840' \
                           f'&EndYear={year}' \
                           f'&optLimit=specDate' \
                           f'&Year={year}' \
                           f'&Month={month}' \
                           f'&Day={day}' \
                           f'&selRowPerPage={interval}' \
                           f'&startRow={start}'
        print(f"Starting at {start} ", end='', flush=True)
        response = requests.get(station_list_url, allow_redirects=True)
        if response.status_code != 200:
            print(f"Server returned error. {response.status_code} {response.content} {station_list_url}")
            exit(1)

        if b'Sorry' in response.content:
            print("Server says data is not available")
            exit(2)

        if b'Your request could not be completed because an error was found' in response.content:
            print("Request error, perhaps the server API has changed?")
            exit(3)

        soup = bs4.BeautifulSoup(response.content, 'html.parser')
        base = soup.find(class_='historical-data-results')
        if base is None:
            print("All stations processed!")
            break

        station_list = base.find_all("form")

        for station in station_list:
            station_id = station.find("input", attrs={'name': "StationID"})['value']
            print(".", end='', flush=True)
            yield station_id
        print("")
        start += interval


In [34]:
def get_station_details(station_id: int, api_host: str = host, year: int = y_year, month: int = y_month,
                        day: int = y_day) -> OrderedDict:
    """Give, a station ID return the meta data."""
    details = OrderedDict()

    details['StationID'] = station_id
    details['url'] = f"{api_host}/climate_data/daily_data_e.html?" \
                     f"&StationID={station_id}" \
                     f"&Prov=" \
                     f"&urlExtension=_e.html" \
                     f"&searchType=stnProv&optLimit=specDate" \
                     f"&StartYear=1840" \
                     f"&EndYear={year}" \
                     f"&selRowPerPage=100" \
                     f"&Line=2" \
                     f"&Month={month}" \
                     f"&Day={day}" \
                     f"&lstProvince=" \
                     f"&timeframe=2" \
                     f"&Year={year}"

    response = requests.get(details['url'])
    assert response.status_code == 200
    soup = bs4.BeautifulSoup(response.content, 'html.parser')

    title_block = soup.select_one("p.table-header").contents
    details['name'] = title_block[0]
    details['province'] = title_block[2]

    for field in ['latitude', 'longitude', 'elevation', 'climateid', 'wmoid', 'tcid']:
        details[field] = soup.select_one(f'div[aria-labelledby={field}]').text.lstrip(' ')
    return details


In [35]:
def main():
    filename = f"canada_weather_station_data_{y_year}-{y_month}-{y_day}.csv"
    print(f'Writing {filename}')
    with open(filename, 'w', newline='') as csv_file:
        first = True
        writer = None
        for station_id in generate_stations():
            details = get_station_details(station_id=station_id)
            if first:
                writer = DictWriter(csv_file, fieldnames=details.keys())
                writer.writeheader()
                first = False
            writer.writerow(details)



In [36]:
if __name__ == '__main__':
    main()

Writing canada_weather_station_data_2026-3-25.csv
Starting at 0 

TypeError: 'NoneType' object is not subscriptable